# sweep-hparam-distribution — ex1: pick the right wandb sweep distribution for each hparam type

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sweep-hparam-distribution`. Running the final beacon cell reports progress against the `Config: sweep hparam distribution` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: sweep hparam distribution` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sweep-hparam-distribution`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sweep-hparam-distribution"
DD_SUBTOPIC = "Config: sweep hparam distribution"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Config: sweep hparam distribution — quick refresher

Inside a wandb sweep's `parameters` block, each hparam picks ONE of these specifiers:

| specifier | when to use | example |
|---|---|---|
| `{'value': X}` | fixed — don't sweep this | `{'value': 'adam'}` |
| `{'values': [...]}` | small discrete set | `{'values': [16, 32, 64]}` |
| `{'distribution': 'uniform', 'min': ..., 'max': ...}` | uniformly on a linear scale | learning rate FOR a model where lr scale is small (rare) |
| `{'distribution': 'log_uniform_values', 'min': ..., 'max': ...}` | uniformly on log scale | LR, weight decay, anything that spans orders of magnitude |
| `{'distribution': 'int_uniform', 'min': ..., 'max': ...}` | integer-valued | num_layers, batch size if continuous |
| `{'distribution': 'categorical', 'values': [...]}` | unordered categories | optimizer name, activation function |

**`log_uniform_values` is the right default for LR.** A linear uniform sample between `1e-5` and `1e-1` wastes ~99% of its samples on the large-LR end (anything above `~1e-2` for most vision tasks is divergent). Log-uniform gives even coverage of the decade scale.

**`uniform` vs `log_uniform_values` is the most common bug.** Picking `uniform` for LR gives you a sweep that's effectively just trying lr in `[5e-2, 1e-1]` with one or two stray samples in the productive range.

**`int_uniform` rounds at sample time** — you don't have to wrap it in `int(...)` yourself.

### Exercise 1 — pick the right wandb sweep distribution for each hparam type

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze hparam types (LR, dropout, num_layers, activation) and select the appropriate wandb sweep distribution specifier for each.
> Keywords: distributions, log-uniform, categorical, int-uniform
> ```

**KCs targeted:** `log-uniform-for-orders-of-magnitude-hparams`, `categorical-vs-values-for-strings`

Implement `ex1_distribution_spec(name)`. Given an hparam name, return the correct wandb-sweep distribution spec dict.

Mapping required:

| name | distribution | spec dict |
|---|---|---|
| `'lr'` | log-uniform value-space | `{'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 1e-1}` |
| `'weight_decay'` | log-uniform value-space | `{'distribution': 'log_uniform_values', 'min': 1e-6, 'max': 1e-2}` |
| `'dropout'` | linear uniform on `[0.0, 0.5]` | `{'distribution': 'uniform', 'min': 0.0, 'max': 0.5}` |
| `'num_layers'` | integer uniform on `[2, 12]` | `{'distribution': 'int_uniform', 'min': 2, 'max': 12}` |
| `'activation'` | categorical from `['relu', 'gelu', 'silu']` | `{'distribution': 'categorical', 'values': ['relu', 'gelu', 'silu']}` |
| `'optimizer'` | discrete (small set, no need for 'categorical') | `{'values': ['sgd', 'adam', 'adamw']}` |

Any other name raises `KeyError`.

The reasoning isn't tested directly — the test checks you produced exactly the right spec dict for each input. The mapping above IS the analysis: which distribution fits which kind of hparam.

Output: `dict`.

In [ ]:
def ex1_distribution_spec(name: str) -> dict:
    """Return the wandb sweep distribution spec for the given hparam name."""
    raise NotImplementedError()


def _test_ex1():
    # === log_uniform_values for LR ===
    spec = ex1_distribution_spec('lr')
    assert spec == {'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 1e-1}, (
        f'lr spec wrong: {spec}; LR spans orders of magnitude → use log_uniform_values'
    )

    # === log_uniform_values for weight_decay (also spans orders of magnitude) ===
    spec = ex1_distribution_spec('weight_decay')
    assert spec == {'distribution': 'log_uniform_values', 'min': 1e-6, 'max': 1e-2}, (
        f'weight_decay spec wrong: {spec}'
    )

    # === linear uniform for dropout (bounded to [0, 0.5], linear scale fine) ===
    spec = ex1_distribution_spec('dropout')
    assert spec == {'distribution': 'uniform', 'min': 0.0, 'max': 0.5}, (
        f'dropout spec wrong: {spec}; bounded linear-scale param → uniform'
    )

    # === int_uniform for num_layers (must be integer) ===
    spec = ex1_distribution_spec('num_layers')
    assert spec == {'distribution': 'int_uniform', 'min': 2, 'max': 12}, (
        f'num_layers spec wrong: {spec}; integer-valued → int_uniform'
    )

    # === categorical for activation (unordered strings) ===
    spec = ex1_distribution_spec('activation')
    assert spec == {'distribution': 'categorical', 'values': ['relu', 'gelu', 'silu']}, (
        f'activation spec wrong: {spec}; unordered string options → categorical'
    )

    # === discrete 'values' for optimizer (small set, no need for the categorical wrapper) ===
    spec = ex1_distribution_spec('optimizer')
    assert spec == {'values': ['sgd', 'adam', 'adamw']}, (
        f'optimizer spec wrong: {spec}'
    )

    # === Unknown name => KeyError ===
    try:
        ex1_distribution_spec('mystery_param')
    except KeyError as e:
        assert 'mystery_param' in str(e), f'KeyError should mention the bad name, got {e!r}'
    else:
        raise AssertionError('expected KeyError for unknown hparam name')

    # === Same call returns equal dicts (no shared mutable state) ===
    s1 = ex1_distribution_spec('lr')
    s2 = ex1_distribution_spec('lr')
    s1['min'] = 999  # mutate one
    assert s2['min'] == 1e-5, 'returned specs must not share mutable state'

    # === All specs are JSON-serializable ===
    import json
    for name in ['lr', 'weight_decay', 'dropout', 'num_layers', 'activation', 'optimizer']:
        _ = json.dumps(ex1_distribution_spec(name))
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_distribution_spec(name):
    table = {
        'lr':           {'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 1e-1},
        'weight_decay': {'distribution': 'log_uniform_values', 'min': 1e-6, 'max': 1e-2},
        'dropout':      {'distribution': 'uniform',            'min': 0.0,  'max': 0.5},
        'num_layers':   {'distribution': 'int_uniform',        'min': 2,    'max': 12},
        'activation':   {'distribution': 'categorical', 'values': ['relu', 'gelu', 'silu']},
        'optimizer':    {'values': ['sgd', 'adam', 'adamw']},
    }
    if name not in table:
        raise KeyError(name)
    # Return a copy so callers can mutate without affecting the table.
    spec = dict(table[name])
    if 'values' in spec and isinstance(spec['values'], list):
        spec['values'] = list(spec['values'])
    return spec
```

**Why LR is log-uniform and dropout is linear.** LR spans 4-5 orders of magnitude in viable ranges (`1e-5` to `1e-1`). Sampling uniformly on the LINEAR scale would waste ~99% of trials on `[1e-2, 1e-1]`. Dropout is bounded to `[0, 0.5]` and varies linearly in its effect — linear uniform is correct.

**Why `int_uniform` for num_layers.** wandb's `int_uniform` rounds at sample time. If you used `uniform` and then `int(...)`-cast in your training script, you'd bias toward the lower bound (a sample of `4.7` becomes `4`, not `5`, by `int()`'s truncation rule).

**`categorical` vs `values`.** Plain `{'values': [...]}` samples uniformly from the list. `{'distribution': 'categorical', 'values': [...]}` is equivalent for equal weights but lets you add a `probabilities` key for non-uniform priors (e.g. weight adam at 0.7, sgd at 0.2, adamw at 0.1). For the activation function we want the option of biasing the prior; for the optimizer the plain form is fine.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()